In [9]:
import gradio as gr
import numpy as np
import cv2

In [10]:
def log_transform(img, c):
    # chuyen anh ve float
    img_float = img.astype(np.float64)

    # tinh log
    log_img = c * np.log(1 + img_float)

    # chuẩn hóa về[0, 255]
    log_img = np.uint8(255 * (log_img) / np.max(log_img))


    return log_img


In [11]:
def gamma_correction(img, gamma, c):

    # chuẩn hóa về [0, 1]
    img_normalized = img / 255.0
    corrected = c * np.power(img_normalized, gamma)

    # chuẩn hóa về [0, 255]
    corrected = np.uint8(corrected * 255)

    return corrected

In [12]:
def equalizeHist(img):
    if len(img.shape) == 3:
        gray_image = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray_image = img.copy() 
    return cv2.equalizeHist(gray_image)

In [21]:
def tienXuLyBSX(img):
  # Handle missing input
  if img is None:
    return None

  
  # Chuyển sang hình xám
  if len(img.shape) == 3: # Check if the image is color
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
  else:
    gray = img.copy()

  # Ensure the image is 8-bit single channel
  if gray.dtype != np.uint8:
      gray = gray.astype(np.uint8)

#  áp dụng clahe để tăng độ tương phản
  clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
  enhanced = clahe.apply(gray)

  # gamma corection để tăng độ sáng
  gamma_corrected = gamma_correction(enhanced, 0.8, 1)

  return gamma_corrected


demo = gr.Interface(fn=tienXuLyBSX, inputs=gr.Image(), 
outputs=gr.Image(),
live=True,
title="Tien xu ly bsx",
description="Tien xu ly bsx")
demo.launch()


* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


In [ ]:
def caiThienAnhVeTinh(img):
    # Handle missing input
    if img is None:
        return None, None, None, None

    

    # Convert to grayscale (Gradio provides RGB)
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        gray = img.copy()

    # Ensure 8-bit single channel
    gray = gray.astype(np.uint8)

    # 1) Log transform to enhance dark regions
    log_enhanced = log_transform(gray, c=1.5)

    # 2) Gamma correction to brighten
    gamma_enhanced = gamma_correction(log_enhanced, gamma=0.8, c=1.0)

    #CLAHE để cải thiện độ tương phản local
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(16, 16))
    final_enhanced = clahe.apply(gamma_enhanced)

    # Return in the same order as outputs
    return gray, log_enhanced, gamma_enhanced, final_enhanced

# Gradio interface with 4 outputs
demo = gr.Interface(
    fn=caiThienAnhVeTinh,
    inputs=gr.Image(type="numpy", label="Input Image"),
    outputs=[
        gr.Image(label="Grayscale"),
        gr.Image(label="Log Enhanced"),
        gr.Image(label="Gamma Corrected"),
        gr.Image(label="CLAHE")
    ],
    live=True,
    title="Cải thiện ảnh vệ tinh",
    description="Log transform → Gamma correction → CLAHE"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [22]:
def caiThienAnhToi(img):
    if img is None:
        return None
    if len(img.shape) == 3: # Check if the image is color
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        gray_img = img.copy()

    gray_img = gray_img.astype(np.uint8)
    # 1. Log transformation để nâng vùng tối
    log_img = log_transform(gray_img, c=2)

    # 2. Gamma correction để điều chỉnh
    gamma_img = gamma_correction(log_img, gamma=0.7, c=1)

    # 3. CLAHE để cải thiện chi tiết
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gamma_img)

    return enhanced

demo = gr.Interface(fn=caiThienAnhToi, inputs=gr.Image(), outputs=gr.Image(),
                    live=True, title="Cải thiện ảnh tối",
                    description="Sử dụng log transform, gamma correction và CLAHE để cải thiện ảnh tối")
demo.launch()



* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.
